# Feature Engineering

This notebook converts the strongest analytical findings into a model-ready feature set. The goal is to preserve the highest-signal fraud features, apply justified transformations to raw variables, and create a small set of engineered interaction features that can improve downstream model performance.


## 1. Objectives

This notebook focuses on the following tasks:

- carry forward the strongest PCA features from the multivariate stage
- apply justified transformations to `Amount` and `Time`
- create multiple interaction features from the strongest PCA signals
- compare original and engineered feature usefulness with respect to the fraud target
- save the engineered dataset and feature list for later outlier analysis, feature selection, and modeling


## Output Guide

- **Base feature table:** shows which variables move forward from the multivariate stage.
- **Engineered feature summary:** records every created feature and the reason it exists.
- **Feature-to-target comparison table:** compares raw and engineered features using the same screening logic.
- **Engineered dataset preview:** confirms the final feature structure before later notebooks use it.
- **Feature importance comparison chart:** helps judge whether the engineered features strengthen the fraud signal.

The purpose of these outputs is to produce a defensible, model-ready dataset rather than create features blindly.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path("..").resolve()))

from src.config import (
    AMOUNT_COLUMN,
    ARTIFACTS_DIR,
    CLEANED_DATA_FILE,
    ENGINEERED_DATA_FILE,
    FEATURE_COLUMNS_FILE,
    PCA_COLUMNS,
    PROJECT_ROOT,
    TARGET_COLUMN,
    TIME_COLUMN,
)
from src.data.data_loader import load_cleaned_data

NOTEBOOK_TABLES_DIR = PROJECT_ROOT / "reports" / "tables" / "08_feature_engineering"
NOTEBOOK_FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "08_feature_engineering"
NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
ENGINEERED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

print("Imports OK")
print(f"  Cleaned data   : {CLEANED_DATA_FILE}")
print(f"  Engineered out : {ENGINEERED_DATA_FILE}")
print(f"  Feature list   : {FEATURE_COLUMNS_FILE}")
print(f"  Tables dir     : {NOTEBOOK_TABLES_DIR}")
print(f"  Figures dir    : {NOTEBOOK_FIGURES_DIR}")


## 2. Load Cleaned Data


In [ ]:
df = load_cleaned_data()

print("Cleaned dataset shape:", df.shape)
df.head()


## 3. Select Base Features

The base feature set follows the multivariate findings: the top 10 PCA features are retained, and `Amount` plus `Time` remain as contextual raw inputs. PCA features are kept unchanged because they are already transformed, centered, and decorrelated by the original dataset construction.


In [ ]:
pca_correlations = df[[*PCA_COLUMNS, TARGET_COLUMN]].corr(numeric_only=True)[TARGET_COLUMN].drop(TARGET_COLUMN)
selected_pca_features = pca_correlations.abs().sort_values(ascending=False).head(10).index.tolist()
base_features = selected_pca_features + [TIME_COLUMN, AMOUNT_COLUMN]

base_feature_table = pd.DataFrame({
    "feature": base_features,
    "feature_type": ["PCA"] * len(selected_pca_features) + ["Raw", "Raw"],
    "correlation_with_class": [pca_correlations[f] for f in selected_pca_features] + [
        df[[TIME_COLUMN, TARGET_COLUMN]].corr(numeric_only=True).loc[TIME_COLUMN, TARGET_COLUMN],
        df[[AMOUNT_COLUMN, TARGET_COLUMN]].corr(numeric_only=True).loc[AMOUNT_COLUMN, TARGET_COLUMN],
    ],
})
base_feature_table["abs_correlation_with_class"] = base_feature_table["correlation_with_class"].abs()
base_feature_table.to_csv(NOTEBOOK_TABLES_DIR / "base_feature_set.csv", index=False)
base_feature_table


## 4. Apply Raw-Feature Transformations

`Amount` and `Time` remain weak standalone predictors, so this notebook transforms them instead of using them only in raw form. The aim is to make their contribution more stable and more useful for downstream modeling.


In [ ]:
engineered_df = df[selected_pca_features + [TIME_COLUMN, AMOUNT_COLUMN, TARGET_COLUMN]].copy()

engineered_df["log_amount"] = np.log1p(engineered_df[AMOUNT_COLUMN])
engineered_df["time_normalized"] = engineered_df[TIME_COLUMN] / engineered_df[TIME_COLUMN].max()
engineered_df["time_day_fraction"] = (engineered_df[TIME_COLUMN] % 86400) / 86400

transformation_summary = pd.DataFrame([
    {"feature": "log_amount", "source": AMOUNT_COLUMN, "transformation": "log1p", "reason": "Reduce right skew and improve model stability"},
    {"feature": "time_normalized", "source": TIME_COLUMN, "transformation": "max scaling", "reason": "Put time on a stable 0-1 scale"},
    {"feature": "time_day_fraction", "source": TIME_COLUMN, "transformation": "modulo day fraction", "reason": "Capture position within an approximate day cycle"},
])
transformation_summary.to_csv(NOTEBOOK_TABLES_DIR / "raw_feature_transformations.csv", index=False)
transformation_summary


## 5. Create Interaction Features

The interaction features below are created from the strongest multivariate PCA signals. They are intentionally limited to a small, justified set so that the feature space grows in a controlled way.


In [ ]:
interaction_definitions = [
    ("V17_V14_interaction", "V17", "V14"),
    ("V17_V12_interaction", "V17", "V12"),
    ("V14_V12_interaction", "V14", "V12"),
    ("V17_V10_interaction", "V17", "V10"),
    ("V17_V16_interaction", "V17", "V16"),
]

interaction_rows = []
for feature_name, feature_a, feature_b in interaction_definitions:
    engineered_df[feature_name] = engineered_df[feature_a] * engineered_df[feature_b]
    interaction_rows.append({
        "engineered_feature": feature_name,
        "feature_a": feature_a,
        "feature_b": feature_b,
        "reason": "Capture joint fraud behavior between high-signal PCA features",
    })

interaction_summary = pd.DataFrame(interaction_rows)
interaction_summary.to_csv(NOTEBOOK_TABLES_DIR / "interaction_feature_summary.csv", index=False)
interaction_summary


## 6. Review Engineered Feature Usefulness

This section compares the screening relationship of the engineered features with the fraud target. These correlation values are not the final feature-selection rule, but they help verify whether the engineered features are reasonable candidates for the next stages.


In [ ]:
engineered_feature_names = [
    "log_amount",
    "time_normalized",
    "time_day_fraction",
    *[row[0] for row in interaction_definitions],
]

engineered_feature_screening = engineered_df[engineered_feature_names + [TARGET_COLUMN]].corr(numeric_only=True)[[TARGET_COLUMN]].drop(index=TARGET_COLUMN)
engineered_feature_screening = engineered_feature_screening.rename(columns={TARGET_COLUMN: "correlation_with_class"}).sort_values("correlation_with_class", key=lambda s: s.abs(), ascending=False)
engineered_feature_screening["abs_correlation_with_class"] = engineered_feature_screening["correlation_with_class"].abs()
engineered_feature_screening.to_csv(NOTEBOOK_TABLES_DIR / "engineered_feature_screening.csv")
engineered_feature_screening


In [ ]:
comparison_rows = []
for feature in selected_pca_features[:5] + [TIME_COLUMN, AMOUNT_COLUMN] + engineered_feature_names:
    corr_value = engineered_df[[feature, TARGET_COLUMN]].corr(numeric_only=True).loc[feature, TARGET_COLUMN]
    comparison_rows.append({
        "feature": feature,
        "correlation_with_class": corr_value,
        "abs_correlation_with_class": abs(corr_value),
        "feature_group": "engineered" if feature in engineered_feature_names else "original",
    })

feature_comparison = pd.DataFrame(comparison_rows).sort_values("abs_correlation_with_class", ascending=False)
feature_comparison.to_csv(NOTEBOOK_TABLES_DIR / "feature_comparison_summary.csv", index=False)
feature_comparison


In [ ]:
plot_df = feature_comparison.head(15).copy()

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="abs_correlation_with_class", y="feature", hue="feature_group")
plt.title("Original vs Engineered Feature Screening Strength")
plt.xlabel("Absolute Correlation with Class")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(NOTEBOOK_FIGURES_DIR / "feature_engineering_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. Build Final Engineered Dataset

The final engineered dataset keeps the selected PCA features, preserves the raw contextual variables, and appends the transformed and interaction-based features created in this notebook.


In [ ]:
final_feature_columns = [
    *selected_pca_features,
    TIME_COLUMN,
    AMOUNT_COLUMN,
    *engineered_feature_names,
]

final_engineered_df = engineered_df[final_feature_columns + [TARGET_COLUMN]].copy()
final_engineered_df.to_csv(ENGINEERED_DATA_FILE, index=False)
FEATURE_COLUMNS_FILE.write_text(json.dumps(final_feature_columns, indent=2), encoding="utf-8")

print("Engineered dataset saved to:", ENGINEERED_DATA_FILE)
print("Feature columns saved to:", FEATURE_COLUMNS_FILE)
print("Final engineered dataset shape:", final_engineered_df.shape)
final_engineered_df.head()


## 8. Engineered Dataset Quality Check


In [ ]:
quality_summary = pd.DataFrame({
    "metric": ["rows", "columns", "missing_values", "duplicate_rows"],
    "value": [
        final_engineered_df.shape[0],
        final_engineered_df.shape[1],
        int(final_engineered_df.isna().sum().sum()),
        int(final_engineered_df.duplicated().sum()),
    ],
})
quality_summary.to_csv(NOTEBOOK_TABLES_DIR / "engineered_dataset_quality_summary.csv", index=False)
quality_summary


## 9. Feature Engineering Interpretation

- The strongest PCA features are retained unchanged because they already represent the most stable and informative transformed signals in the dataset.
- `Amount` and `Time` are not discarded; instead, they are converted into more model-friendly forms through log and scale-based transformations.
- Multiple interaction features are created because fraud behavior is unlikely to be explained by one high-signal PCA component alone.
- The interaction set is intentionally small and targeted so that later notebooks can evaluate usefulness without creating unnecessary feature explosion.


## 10. Connection to Modeling and Decision System

- The engineered feature set should improve the fraud-risk model by combining strong PCA features with transformed raw variables and a small number of high-value interactions.
- Interaction features such as `V17_V14_interaction`, `V17_V12_interaction`, and `V17_V10_interaction` can help the model capture joint fraud signatures that may strengthen `BLOCK` and `REVIEW` decisions.
- Transformed raw features such as `log_amount` and `time_normalized` make it easier for baseline linear models to use contextual information without being dominated by scale problems.
- The final engineered dataset is designed to support both linear baselines and more flexible tree-based models before the later feature-selection stage narrows the final set.


## 11. Key Insights

- Feature engineering should focus on a controlled set of high-signal additions rather than creating many untested derived variables.
- The top PCA features remain the core predictive signals, while transformed raw features and interactions act as supporting enhancements.
- Multiple interaction features are justified because the multivariate stage suggested that fraud behavior follows structured, non-linear feature relationships.
- The engineered dataset created here is the correct input for the next notebooks on outlier analysis and feature selection.


## 12. Next Step

The next notebook should be `09_outlier_analysis.ipynb`. It should perform a dedicated outlier review on the engineered dataset, with special attention to transformed raw variables and the newly created interaction features.


In [ ]:
feature_engineering_report = f"""# Feature Engineering Report

## Key Findings

- The engineered dataset keeps the strongest PCA features unchanged and adds transformed raw features plus a small set of interaction features.
- `Amount` and `Time` are retained in raw form but are also converted into more model-friendly variants.
- Multiple interaction features are created because the strongest fraud signals are likely to interact rather than act independently.

## Feature Engineering Interpretation

- The strongest PCA features are retained unchanged because they already represent the most stable and informative transformed signals in the dataset.
- `Amount` and `Time` are not discarded; instead, they are converted into more model-friendly forms through log and scale-based transformations.
- Multiple interaction features are created because fraud behavior is unlikely to be explained by one high-signal PCA component alone.
- The interaction set is intentionally small and targeted so that later notebooks can evaluate usefulness without creating unnecessary feature explosion.

## Connection to Modeling and Decision System

- The engineered feature set should improve the fraud-risk model by combining strong PCA features with transformed raw variables and a small number of high-value interactions.
- Interaction features such as `V17_V14_interaction`, `V17_V12_interaction`, and `V17_V10_interaction` can help the model capture joint fraud signatures that may strengthen `BLOCK` and `REVIEW` decisions.
- Transformed raw features such as `log_amount` and `time_normalized` make it easier for baseline linear models to use contextual information without being dominated by scale problems.
- The final engineered dataset is designed to support both linear baselines and more flexible tree-based models before the later feature-selection stage narrows the final set.

## Key Insights

- Feature engineering should focus on a controlled set of high-signal additions rather than creating many untested derived variables.
- The top PCA features remain the core predictive signals, while transformed raw features and interactions act as supporting enhancements.
- Multiple interaction features are justified because the multivariate stage suggested that fraud behavior follows structured, non-linear feature relationships.
- The engineered dataset created here is the correct input for the next notebooks on outlier analysis and feature selection.

## Saved Tables

- `reports/tables/08_feature_engineering/base_feature_set.csv`
- `reports/tables/08_feature_engineering/raw_feature_transformations.csv`
- `reports/tables/08_feature_engineering/interaction_feature_summary.csv`
- `reports/tables/08_feature_engineering/engineered_feature_screening.csv`
- `reports/tables/08_feature_engineering/feature_comparison_summary.csv`
- `reports/tables/08_feature_engineering/engineered_dataset_quality_summary.csv`
- `reports/tables/08_feature_engineering/feature_engineering_report.md`

## Saved Figures

- `reports/figures/08_feature_engineering/feature_engineering_comparison.png`

## Saved Data Assets

- `{ENGINEERED_DATA_FILE}`
- `{FEATURE_COLUMNS_FILE}`
"""

report_path = NOTEBOOK_TABLES_DIR / "feature_engineering_report.md"
report_path.write_text(feature_engineering_report, encoding="utf-8")
print(f"Feature engineering report saved to: {report_path}")
